# Torus Geometry: Agent Initialization

Explore agent generation on a flat toroidal space $[0, 1)^d$ with wrap-around distance.

**Key questions:**
- How does dimensionality affect pairwise distance distributions?
- At what dimension does concentration of measure make distance-based tie formation degenerate?
- How do different initialization schemes (uniform vs. Gaussian mixture) affect initial network structure?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
import networkx as nx
from typing import NamedTuple

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120

RNG = np.random.default_rng(42)

## 1. Toroidal Distance

On a torus $[0,1)^d$, the distance between two points wraps around:

$$d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{i=1}^{d} \min(|x_i - y_i|, 1 - |x_i - y_i|)^2}$$

This eliminates edge effects — every position is equivalent.

In [ ]:
def torus_distance_matrix(positions: np.ndarray) -> np.ndarray:
    """Compute pairwise toroidal distances for agents in [0, 1)^d."""
    n = positions.shape[0]
    diff = positions[:, np.newaxis, :] - positions[np.newaxis, :, :]
    # Wrap-around: take the shorter path
    diff = np.abs(diff)
    diff = np.minimum(diff, 1.0 - diff)
    return np.sqrt(np.sum(diff ** 2, axis=2))

## 2. Agent Generation: Uniform

Simplest case — agents distributed uniformly on the torus. No pre-existing structure.

In [ ]:
def generate_uniform_torus(n: int, d: int, rng: np.random.Generator) -> np.ndarray:
    """Generate n agents uniformly on [0, 1)^d torus."""
    return rng.uniform(0, 1, size=(n, d))

In [ ]:
N = 300

# Generate agents in 2D for visualization
agents_2d = generate_uniform_torus(N, 2, RNG)

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.scatter(agents_2d[:, 0], agents_2d[:, 1], s=10, alpha=0.6)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.set_title('Uniform agents on 2D torus')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
plt.tight_layout()
plt.show()

## 3. Agent Generation: Gaussian Mixture

Place cluster centers on the torus, then sample agents from wrapped Gaussians around each center.
This gives pre-existing community structure.

In [ ]:
def generate_gmm_torus(
    n: int,
    d: int,
    n_clusters: int,
    sigma: float,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    """Generate n agents from a Gaussian mixture on the torus.
    
    Returns (positions, cluster_labels).
    """
    # Place cluster centers uniformly
    centers = rng.uniform(0, 1, size=(n_clusters, d))
    
    # Assign agents to clusters (equal size)
    labels = rng.integers(0, n_clusters, size=n)
    
    # Sample positions from wrapped Gaussians
    positions = centers[labels] + rng.normal(0, sigma, size=(n, d))
    positions = positions % 1.0  # Wrap to [0, 1)
    
    return positions, labels

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, sigma in zip(axes, [0.03, 0.08, 0.15]):
    positions, labels = generate_gmm_torus(N, 2, 5, sigma, np.random.default_rng(42))
    scatter = ax.scatter(positions[:, 0], positions[:, 1], c=labels, cmap='tab10', s=10, alpha=0.6)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.set_title(f'GMM, $\\sigma$ = {sigma}')

plt.suptitle('Gaussian Mixture on 2D Torus (5 clusters)', y=1.02)
plt.tight_layout()
plt.show()

## 4. Concentration of Measure: Dimensionality Experiment

As dimension $d$ grows, pairwise distances concentrate around their mean.
This is the key test: at what $d$ does the distance distribution collapse?

In [ ]:
dimensions = [2, 5, 10, 20, 50, 100]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

stats = []
for ax, d in zip(axes.flat, dimensions):
    agents = generate_uniform_torus(N, d, np.random.default_rng(42))
    dist_matrix = torus_distance_matrix(agents)
    # Extract upper triangle (unique pairs)
    dists = dist_matrix[np.triu_indices(N, k=1)]
    
    mean, std = dists.mean(), dists.std()
    cv = std / mean  # Coefficient of variation
    stats.append({'d': d, 'mean': mean, 'std': std, 'cv': cv})
    
    ax.hist(dists, bins=60, density=True, alpha=0.7, color='steelblue')
    ax.axvline(mean, color='red', linestyle='--', label=f'mean={mean:.3f}')
    ax.set_title(f'd = {d}  (CV = {cv:.3f})')
    ax.set_xlabel('Distance')
    ax.legend(fontsize=8)

plt.suptitle(f'Pairwise Distance Distributions on Torus (N={N})', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Summary: coefficient of variation vs dimension
dims = [s['d'] for s in stats]
cvs = [s['cv'] for s in stats]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dims, cvs, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Dimension $d$')
ax.set_ylabel('Coefficient of Variation (std/mean)')
ax.set_title('Distance Concentration on Torus')
ax.axhline(0.05, color='red', linestyle='--', alpha=0.5, label='CV = 0.05 (near-degenerate)')
ax.legend()
ax.set_xscale('log')
plt.tight_layout()
plt.show()

print('Dimension | Mean Distance | Std  | CV')
print('-' * 45)
for s in stats:
    print(f"  d={s['d']:>3}   |    {s['mean']:.4f}    | {s['std']:.4f} | {s['cv']:.4f}")

## 5. Tie Formation: Distance Threshold

Form initial ties using a simple rule: connect agents within distance $r$.
This lets us see what network structure each initialization produces.

We calibrate $r$ to target a mean degree $\bar{k} \approx 10$ (typical for social networks).

In [ ]:
def form_network_threshold(dist_matrix: np.ndarray, radius: float) -> nx.Graph:
    """Form network by connecting all pairs within distance radius."""
    n = dist_matrix.shape[0]
    G = nx.Graph()
    G.add_nodes_from(range(n))
    mask = (dist_matrix < radius) & (dist_matrix > 0)
    edges = list(zip(*np.where(np.triu(mask, k=1))))
    G.add_edges_from(edges)
    return G


def calibrate_radius(dist_matrix: np.ndarray, target_mean_degree: float) -> float:
    """Find radius that gives approximately the target mean degree."""
    n = dist_matrix.shape[0]
    dists = dist_matrix[np.triu_indices(n, k=1)]
    # Mean degree = (n-1) * P(distance < r)
    # So P(distance < r) = target_mean_degree / (n-1)
    target_quantile = target_mean_degree / (n - 1)
    return np.quantile(dists, target_quantile)

In [ ]:
def network_summary(G: nx.Graph) -> dict:
    """Compute key network statistics."""
    degrees = [d for _, d in G.degree()]
    return {
        'n_nodes': G.number_of_nodes(),
        'n_edges': G.number_of_edges(),
        'mean_degree': np.mean(degrees),
        'std_degree': np.std(degrees),
        'clustering': nx.average_clustering(G),
        'components': nx.number_connected_components(G),
    }

### 5a. Uniform initialization — network at different dimensions

In [ ]:
TARGET_K = 10
test_dims = [2, 5, 10, 20]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, d in enumerate(test_dims):
    rng_d = np.random.default_rng(42)
    agents = generate_uniform_torus(N, d, rng_d)
    dist_mat = torus_distance_matrix(agents)
    r = calibrate_radius(dist_mat, TARGET_K)
    G = form_network_threshold(dist_mat, r)
    summary = network_summary(G)
    
    # Top row: degree distribution
    degrees = [deg for _, deg in G.degree()]
    axes[0, i].hist(degrees, bins=range(0, max(degrees) + 2), density=True, alpha=0.7, color='steelblue')
    axes[0, i].set_title(f'd={d}: $\\bar{{k}}$={summary["mean_degree"]:.1f}, CC={summary["clustering"]:.3f}')
    axes[0, i].set_xlabel('Degree')
    
    # Bottom row: network visualization (2D projection)
    if d == 2:
        pos = {j: agents[j, :2] for j in range(N)}
    else:
        pos = {j: agents[j, :2] for j in range(N)}  # Project to first 2 dims
    
    nx.draw_networkx_nodes(G, pos, ax=axes[1, i], node_size=8, node_color='steelblue', alpha=0.5)
    nx.draw_networkx_edges(G, pos, ax=axes[1, i], alpha=0.05, width=0.5)
    axes[1, i].set_xlim(0, 1)
    axes[1, i].set_ylim(0, 1)
    axes[1, i].set_aspect('equal')

axes[0, 0].set_ylabel('Density')
axes[1, 0].set_ylabel('Spatial layout')
plt.suptitle(f'Uniform Torus Networks (N={N}, target $\\bar{{k}}$={TARGET_K})', y=1.02)
plt.tight_layout()
plt.show()

### 5b. GMM initialization — effect of cluster separation

In [ ]:
sigmas = [0.03, 0.06, 0.10, 0.20]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, sigma in enumerate(sigmas):
    rng_s = np.random.default_rng(42)
    agents, labels = generate_gmm_torus(N, 2, 5, sigma, rng_s)
    dist_mat = torus_distance_matrix(agents)
    r = calibrate_radius(dist_mat, TARGET_K)
    G = form_network_threshold(dist_mat, r)
    summary = network_summary(G)
    
    # Top row: degree distribution
    degrees = [deg for _, deg in G.degree()]
    axes[0, i].hist(degrees, bins=range(0, max(degrees) + 2), density=True, alpha=0.7, color='steelblue')
    axes[0, i].set_title(f'$\\sigma$={sigma}: CC={summary["clustering"]:.3f}')
    axes[0, i].set_xlabel('Degree')
    
    # Bottom row: spatial layout colored by cluster
    pos = {j: agents[j] for j in range(N)}
    node_colors = [plt.cm.tab10(labels[j]) for j in range(N)]
    nx.draw_networkx_nodes(G, pos, ax=axes[1, i], node_size=8, node_color=node_colors, alpha=0.6)
    nx.draw_networkx_edges(G, pos, ax=axes[1, i], alpha=0.05, width=0.5)
    axes[1, i].set_xlim(0, 1)
    axes[1, i].set_ylim(0, 1)
    axes[1, i].set_aspect('equal')

plt.suptitle(f'GMM Torus Networks (N={N}, 5 clusters, target $\\bar{{k}}$={TARGET_K})', y=1.02)
plt.tight_layout()
plt.show()

## 6. Structural Hole Metrics at t=0

Measure Burt's constraint and betweenness centrality on the initial networks.
This gives us a baseline: how many structural holes exist before any dynamics run?

In [ ]:
def burt_constraint(G: nx.Graph) -> dict[int, float]:
    """Compute Burt's constraint for each node.
    
    Lower constraint = more structural holes = better brokerage position.
    C_i = sum_j (p_ij + sum_q p_iq * p_qj)^2
    where p_ij = strength of i's tie to j as proportion of i's total ties.
    """
    constraint = {}
    for i in G.nodes():
        neighbors_i = set(G.neighbors(i))
        if len(neighbors_i) == 0:
            constraint[i] = np.nan
            continue
        
        deg_i = len(neighbors_i)
        c_i = 0.0
        for j in neighbors_i:
            # Direct proportion
            p_ij = 1.0 / deg_i
            # Indirect: sum over shared neighbors
            indirect = 0.0
            for q in neighbors_i:
                if q != j and G.has_edge(q, j):
                    p_iq = 1.0 / deg_i
                    neighbors_q = set(G.neighbors(q))
                    p_qj = 1.0 / len(neighbors_q) if j in neighbors_q else 0.0
                    indirect += p_iq * p_qj
            c_i += (p_ij + indirect) ** 2
        constraint[i] = c_i
    return constraint

In [ ]:
# Compare: uniform 2D vs GMM 2D (tight clusters)
configs = [
    ('Uniform 2D', generate_uniform_torus(N, 2, np.random.default_rng(42)), None),
]
gmm_pos, gmm_labels = generate_gmm_torus(N, 2, 5, 0.05, np.random.default_rng(42))
configs.append(('GMM 2D ($\\sigma$=0.05)', gmm_pos, gmm_labels))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for col, (name, agents, labels) in enumerate(configs):
    dist_mat = torus_distance_matrix(agents)
    r = calibrate_radius(dist_mat, TARGET_K)
    G = form_network_threshold(dist_mat, r)
    
    constraint = burt_constraint(G)
    betweenness = nx.betweenness_centrality(G)
    
    c_vals = [v for v in constraint.values() if not np.isnan(v)]
    b_vals = list(betweenness.values())
    
    axes[0, col].hist(c_vals, bins=40, density=True, alpha=0.7, color='coral')
    axes[0, col].set_title(f'{name}\nBurt\'s Constraint')
    axes[0, col].set_xlabel('Constraint (lower = more holes)')
    
    axes[1, col].hist(b_vals, bins=40, density=True, alpha=0.7, color='seagreen')
    axes[1, col].set_title(f'Betweenness Centrality')
    axes[1, col].set_xlabel('Betweenness')

plt.suptitle(f'Structural Hole Metrics at t=0 (N={N})', y=1.02)
plt.tight_layout()
plt.show()

for name, agents, labels in configs:
    dist_mat = torus_distance_matrix(agents)
    r = calibrate_radius(dist_mat, TARGET_K)
    G = form_network_threshold(dist_mat, r)
    c = burt_constraint(G)
    c_vals = [v for v in c.values() if not np.isnan(v)]
    print(f'{name}: mean constraint = {np.mean(c_vals):.4f}, std = {np.std(c_vals):.4f}')

## Next Steps

- Compare these baseline metrics against the hyperbolic geometry notebook
- Add probabilistic (Fermi-Dirac) tie formation instead of hard threshold
- Implement temporal dynamics: attention budgets, tie decay, triadic closure